In [0]:
# =============================================================================
# LANDING ZONE SYNTHETIC DATA GENERATOR (Databricks / PySpark)
# =============================================================================
# Purpose:
#   Simulates a real fintech source system landing into a multi-format landing
#   zone. Designed to be run once with IS_DAILY_INCREMENTAL = False, then run
#   every day after with IS_DAILY_INCREMENTAL = True (e.g. as a Databricks Job
#   on a daily schedule feeding a dbt "full refresh" on top of the landing
#   layer, per your bronze -> dbt silver/gold flow).
#
# Design principles applied from your requirements:
#   1. Volume:            ~100k customers / 150k accounts scale up over time
#                          via daily incremental growth, not just one big load.
#   2. Raw only:           NO fraud flags, NO computed risk scores, NO computed
#                          commission_amount / revenue. Those are dbt's job.
#                          This layer only stores rates and raw events.
#   3. Multi-format:       Parquet (customers, accounts, commission_rules),
#                          CSV (merchants, billers), JSON (transactions).
#   4. Daily incremental:  New rows land each day (new customers, new
#                          transactions) AND some existing rows are updated
#                          (profile changes, pending -> settled/declined,
#                          commission rate changes) - a proper CDC-style feed.
#   5. Domain-logical nulls only: card_number is null unless the channel uses
#                          a card, ip_address is null for ATM, biller_reference
#                          is null unless it's a bill payment, etc. Never random
#                          corruption.
#   6. Transaction lifecycle: transactions land PENDING and transition to
#                          SETTLED / DECLINED / CANCELLED on later runs.
#
# New in this version:
#   - transactions are now polymorphic across 3 real transaction_types:
#       'purchase'      -> merchant_id populated (card present)
#       'bill_payment'  -> biller_id populated (biller_reference populated)
#       'transfer'      -> destination_account_id populated (account to account)
#     Exactly one "counterparty" column is populated per row, the others null.
#   - billers dimension table (utilities, telecom, insurance, education...).
#   - commission_rules table modeled as SCD2 (effective_start/end_date,
#     is_current). Each day a subset of active rules are "closed" and
#     re-inserted with a new commission_value effective that day - so the
#     table changes incrementally, not in bulk, exactly like a real fee
#     schedule maintained by a pricing team. dbt joins transactions to the
#     rule that was current on transaction_date to compute revenue.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.types import StructType, StructField, ArrayType, IntegerType, StringType, DoubleType
import datetime

# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------
LANDING_PATH = "/Volumes/main/default/landing_zone"
RUN_DATE = datetime.date.today()
RUN_DATE_STR = RUN_DATE.strftime("%Y%m%d")

dbutils.widgets.dropdown("is_incremental", "false", ["true", "false"])
IS_DAILY_INCREMENTAL = dbutils.widgets.get("is_incremental") == "true"

# Base (initial-load) volumes
NUM_CUSTOMERS = 100_000
NUM_ACCOUNTS = 150_000
NUM_MERCHANTS = 5_000
NUM_BILLERS = 300
NUM_INITIAL_TX_DAYS_BACKFILL = 1_000_000  # historical purchase/transfer/bill mix, one-time backfill

# Daily incremental volumes (tune these to taste)
NEW_CUSTOMERS_PER_DAY = 313
NEW_ACCOUNTS_PER_DAY = 398
DAILY_TX_VOLUME = 59823

# Mix of transaction types generated each day
TX_TYPE_WEIGHTS = {"purchase": 0.65, "bill_payment": 0.20, "transfer": 0.15}

# Commission table churn: fraction of *currently active* rules that get
# re-priced on any given incremental run (NOT all of them)
COMMISSION_DAILY_CHANGE_FRACTION = 0.12


def path(sub):
    return f"{LANDING_PATH}/{sub}"


def landing_exists(sub):
    try:
        dbutils.fs.ls(path(sub))
        return True
    except Exception:
        return False


# =============================================================================
# 1. CUSTOMERS (Parquet) - initial batch + daily new signups + profile updates
# =============================================================================
def generate_customers():
    if not IS_DAILY_INCREMENTAL:
        df = spark.range(1, NUM_CUSTOMERS + 1) \
            .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lpad(F.col("id"), 8, "0"))) \
            .withColumn("first_name", F.elt((F.rand() * 5 + 1).cast("int"),
                        F.lit("John"), F.lit("Jane"), F.lit("Alex"), F.lit("Emily"), F.lit("Michael"))) \
            .withColumn("last_name", F.elt((F.rand() * 5 + 1).cast("int"),
                        F.lit("Smith"), F.lit("Doe"), F.lit("Johnson"), F.lit("Brown"), F.lit("Taylor"))) \
            .withColumn("primary_email", F.lower(F.concat(F.col("first_name"), F.lit("."), F.col("last_name"),
                        F.lit("@email.com")))) \
            .withColumn("secondary_email", F.when(F.rand() > 0.7,
                        F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
            .withColumn("phone_number", F.when(F.rand() > 0.1,
                        F.concat(F.lit("+1"), (F.rand() * 9_000_000_000 + 1_000_000_000).cast("long"))).otherwise(F.lit(None))) \
            .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                        F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
            .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
            .withColumn("kyc_status", F.when(F.rand() > 0.05, "VERIFIED").otherwise("PENDING")) \
            .withColumn("signup_date", F.date_add(F.to_date(F.lit("2021-01-01")), (F.rand() * 1000).cast("int"))) \
            .withColumn("created_at", F.to_timestamp(F.lit(str(RUN_DATE)))) \
            .withColumn("updated_at", F.to_timestamp(F.lit(str(RUN_DATE)))) \
            .drop("id")
        df.write.format("parquet").mode("overwrite").save(path("customers"))
        return

    # --- Incremental: new signups today ---
    df_new = spark.range(1, NEW_CUSTOMERS_PER_DAY + 1) \
        .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lit(RUN_DATE_STR + "_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("first_name", F.elt((F.rand() * 5 + 1).cast("int"),
                    F.lit("John"), F.lit("Jane"), F.lit("Alex"), F.lit("Emily"), F.lit("Michael"))) \
        .withColumn("last_name", F.elt((F.rand() * 5 + 1).cast("int"),
                    F.lit("Smith"), F.lit("Doe"), F.lit("Johnson"), F.lit("Brown"), F.lit("Taylor"))) \
        .withColumn("primary_email", F.lower(F.concat(F.col("first_name"), F.lit("."), F.col("last_name"),
                    F.lit("@email.com")))) \
        .withColumn("secondary_email", F.when(F.rand() > 0.7,
                    F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
        .withColumn("phone_number", F.when(F.rand() > 0.1,
                    F.concat(F.lit("+1"), (F.rand() * 9_000_000_000 + 1_000_000_000).cast("long"))).otherwise(F.lit(None))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
        .withColumn("kyc_status", F.when(F.rand() > 0.15, "VERIFIED").otherwise("PENDING")) \
        .withColumn("signup_date", F.lit(RUN_DATE)) \
        .withColumn("created_at", F.current_timestamp()) \
        .withColumn("updated_at", F.current_timestamp()) \
        .drop("id")

    # --- Incremental: profile updates for a slice of existing customers ---
    df_updates = None
    if landing_exists("customers"):
        df_existing = spark.read.parquet(path("customers"))
        df_updates = df_existing.sample(withReplacement=False, fraction=0.02) \
            .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
            .withColumn("secondary_email", F.when(F.rand() > 0.7,
                        F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
            .withColumn("kyc_status", F.when(F.rand() > 0.02, "VERIFIED").otherwise("PENDING")) \
            .withColumn("updated_at", F.current_timestamp())

    df_out = df_new if df_updates is None else df_new.unionByName(df_updates)
    df_out.write.format("parquet").mode("append").save(path("customers"))


# =============================================================================
# 2. ACCOUNTS (Parquet) - initial batch + daily new accounts for new customers
# =============================================================================
def generate_accounts():
    if not IS_DAILY_INCREMENTAL:
        df = spark.range(1, NUM_ACCOUNTS + 1) \
            .withColumn("account_id", F.concat(F.lit("ACC_"), F.lpad(F.col("id"), 8, "0"))) \
            .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lpad((F.rand() * NUM_CUSTOMERS + 1).cast("int"), 8, "0"))) \
            .withColumn("account_type", F.when(F.rand() > 0.3, "Checking").otherwise("Savings")) \
            .withColumn("account_status", F.elt((F.rand() * 4 + 1).cast("int"),
                        F.lit("Active"), F.lit("Active"), F.lit("Active"), F.lit("Dormant"))) \
            .withColumn("currency", F.lit("USD")) \
            .withColumn("created_at", F.to_timestamp(F.date_add(F.to_date(F.lit("2021-06-01")), (F.rand() * 800).cast("int")))) \
            .withColumn("updated_at", F.to_timestamp(F.lit(str(RUN_DATE)))) \
            .drop("id")
        df.write.format("parquet").mode("overwrite").save(path("accounts"))
        return

    # link new accounts to a random existing customer (old or new-today)
    df_customers = spark.read.parquet(path("customers")).select("customer_id")
    df_new = spark.range(1, NEW_ACCOUNTS_PER_DAY + 1) \
        .withColumn("account_id", F.concat(F.lit("ACC_"), F.lit(RUN_DATE_STR + "_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("account_type", F.when(F.rand() > 0.3, "Checking").otherwise("Savings")) \
        .withColumn("account_status", F.lit("Active")) \
        .withColumn("currency", F.lit("USD")) \
        .withColumn("created_at", F.current_timestamp()) \
        .withColumn("updated_at", F.current_timestamp()) \
        .drop("id")
    # attach a random existing customer_id per new account via row_number join
    from pyspark.sql.window import Window
    w = Window.orderBy(F.monotonically_increasing_id())
    df_cust_sample = df_customers.orderBy(F.rand()).limit(NEW_ACCOUNTS_PER_DAY) \
        .withColumn("rn", F.row_number().over(w))
    df_new = df_new.withColumn("rn", F.row_number().over(w))
    df_new = df_new.join(df_cust_sample, on="rn", how="inner").drop("rn")

    # small slice of status changes on existing accounts (Active <-> Dormant)
    df_status_updates = None
    if landing_exists("accounts"):
        df_existing = spark.read.parquet(path("accounts"))
        df_status_updates = df_existing.sample(withReplacement=False, fraction=0.01) \
            .withColumn("account_status", F.when(F.col("account_status") == "Active", "Dormant").otherwise("Active")) \
            .withColumn("updated_at", F.current_timestamp())

    df_out = df_new if df_status_updates is None else df_new.unionByName(df_status_updates)
    df_out.write.format("parquet").mode("append").save(path("accounts"))


# =============================================================================
# 3. MERCHANTS (CSV) - static-ish reference data
# =============================================================================
def generate_merchants():
    if IS_DAILY_INCREMENTAL:
        return  # merchants onboard rarely; skip on daily runs to keep this reliable/simple
    df = spark.range(1, NUM_MERCHANTS + 1) \
        .withColumn("merchant_id", F.concat(F.lit("MERCH_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("merchant_name", F.concat(F.lit("Store_"), F.col("id"))) \
        .withColumn("mcc_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("5411"), F.lit("5812"), F.lit("5732"), F.lit("5999"))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .withColumn("onboarded_at", F.date_add(F.to_date(F.lit("2020-01-01")), (F.rand() * 1500).cast("int"))) \
        .drop("id")
    df.write.format("csv").option("header", "true").mode("overwrite").save(path("merchants"))


# =============================================================================
# 4. BILLERS (CSV) - new dimension for bill_payment transactions
# =============================================================================
def generate_billers():
    if IS_DAILY_INCREMENTAL:
        return  # stable reference data
    df = spark.range(1, NUM_BILLERS + 1) \
        .withColumn("biller_id", F.concat(F.lit("BILL_"), F.lpad(F.col("id"), 5, "0"))) \
        .withColumn("biller_name", F.concat(F.lit("Biller_"), F.col("id"))) \
        .withColumn("biller_category", F.elt((F.rand() * 6 + 1).cast("int"),
                    F.lit("Electricity"), F.lit("Water"), F.lit("Internet"),
                    F.lit("Mobile_Topup"), F.lit("Insurance"), F.lit("Education"))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .drop("id")
    df.write.format("csv").option("header", "true").mode("overwrite").save(path("billers"))


# =============================================================================
# 5. COMMISSION RULES (Parquet, mutable CURRENT STATE) - drives revenue in dbt
# =============================================================================
# This table intentionally has NO effective_start/end_date or is_current
# columns - it represents the source system's current fee schedule, same as
# a real upstream pricing table. History/versioning is dbt's job: point a
# `dbt snapshot` (strategy: check, check_cols: [commission_value]; unique_key:
# commission_rule_id) at this table and dbt will build the SCD2 history
# (dbt_valid_from / dbt_valid_to) for you on every run.
#
# Each day, ~COMMISSION_DAILY_CHANGE_FRACTION of rows get a new
# commission_value (a real pricing team never reprices everything at once) -
# the rest of the table is written back unchanged so dbt snapshot sees a
# stable natural key with only some rows different.
COMMISSION_SCOPES = [
    ("purchase", "5411"), ("purchase", "5812"), ("purchase", "5732"), ("purchase", "5999"),
    ("bill_payment", "Electricity"), ("bill_payment", "Water"), ("bill_payment", "Internet"),
    ("bill_payment", "Mobile_Topup"), ("bill_payment", "Insurance"), ("bill_payment", "Education"),
    ("transfer", "internal"),
]


def generate_commission_rules():
    if not IS_DAILY_INCREMENTAL:
        rows = []
        for tx_type, scope_value in COMMISSION_SCOPES:
            rows.append((
                f"RULE_{tx_type}_{scope_value}",
                tx_type,
                scope_value,
                "percentage" if tx_type != "transfer" else "flat",
                float(round(__import__("random").uniform(0.010, 0.035), 4)) if tx_type != "transfer" else 0.50,
            ))
        schema = StructType([
            StructField("commission_rule_id", StringType()),
            StructField("scope_type", StringType()),
            StructField("scope_value", StringType()),
            StructField("commission_type", StringType()),
            StructField("commission_value", DoubleType()),
        ])
        df = spark.createDataFrame(rows, schema) \
            .withColumn("updated_at", F.current_timestamp())
        df.write.format("parquet").mode("overwrite").save(path("commission_rules"))
        return

    # --- Incremental: re-price a subset of rows, write the full table back ---
    df_existing = spark.read.parquet(path("commission_rules"))

    df_out = df_existing \
        .withColumn("_reprice", F.rand() < COMMISSION_DAILY_CHANGE_FRACTION) \
        .withColumn("commission_value", F.when(F.col("_reprice"),
                    F.round(F.col("commission_value") * (F.lit(1) + (F.rand() * 0.4 - 0.2)), 4))
                    .otherwise(F.col("commission_value"))) \
        .withColumn("updated_at", F.when(F.col("_reprice"), F.current_timestamp())
                    .otherwise(F.col("updated_at"))) \
        .drop("_reprice")

    # overwrite = this table always reflects the source system's CURRENT
    # state; dbt snapshot is what preserves history across runs.
    df_out.write.format("parquet").mode("overwrite").save(path("commission_rules"))


# =============================================================================
# 6. TRANSACTIONS (JSON) - polymorphic purchase / bill_payment / transfer
#    + daily lifecycle transitions for yesterday's PENDING rows
# =============================================================================
def _new_transactions(df_accounts, df_merchants, df_billers, n_rows, id_prefix_seed):
    df = spark.range(1, n_rows + 1) \
        .withColumn("transaction_id", F.concat(F.lit("TX_"), F.lit(id_prefix_seed + "_"), F.lpad(F.col("id"), 8, "0"))) \
        .withColumn("_r", F.rand())

    df = df.withColumn(
        "transaction_type",
        F.when(F.col("_r") < TX_TYPE_WEIGHTS["purchase"], "purchase")
         .when(F.col("_r") < TX_TYPE_WEIGHTS["purchase"] + TX_TYPE_WEIGHTS["bill_payment"], "bill_payment")
         .otherwise("transfer")
    ).drop("_r")

    n_accounts = df_accounts.count()
    n_merchants = df_merchants.count()
    n_billers = df_billers.count()

    df = df.withColumn("account_id", F.concat(F.lit("ACC_"), F.lpad((F.rand() * n_accounts + 1).cast("int"), 8, "0")))

    # counterparty columns - only ONE populated per row, by transaction_type
    df = df \
        .withColumn("merchant_id", F.when(F.col("transaction_type") == "purchase",
                    F.concat(F.lit("MERCH_"), F.lpad((F.rand() * n_merchants + 1).cast("int"), 6, "0"))).otherwise(F.lit(None))) \
        .withColumn("biller_id", F.when(F.col("transaction_type") == "bill_payment",
                    F.concat(F.lit("BILL_"), F.lpad((F.rand() * n_billers + 1).cast("int"), 5, "0"))).otherwise(F.lit(None))) \
        .withColumn("destination_account_id", F.when(F.col("transaction_type") == "transfer",
                    F.concat(F.lit("ACC_"), F.lpad((F.rand() * n_accounts + 1).cast("int"), 8, "0"))).otherwise(F.lit(None))) \
        .withColumn("biller_reference_number", F.when(F.col("transaction_type") == "bill_payment",
                    F.concat(F.lit("REF"), (F.rand() * 900000000 + 100000000).cast("long"))).otherwise(F.lit(None)))

    # amount ranges differ by type
    df = df.withColumn("amount",
        F.when(F.col("transaction_type") == "purchase", F.round(F.rand() * 1200 + 1.50, 2))
         .when(F.col("transaction_type") == "bill_payment", F.round(F.rand() * 300 + 10.00, 2))
         .otherwise(F.round(F.rand() * 5000 + 5.00, 2))
    )

    # channel differs by type: ATM only makes sense for purchases
    df = df.withColumn("channel",
        F.when(F.col("transaction_type") == "purchase",
               F.elt((F.rand() * 4 + 1).cast("int"), F.lit("Web"), F.lit("Mobile"), F.lit("POS"), F.lit("ATM")))
         .otherwise(F.elt((F.rand() * 2 + 1).cast("int"), F.lit("Web"), F.lit("Mobile")))
    )

    # domain-logical nulls: card only present for card-based purchase channels
    df = df.withColumn("card_number",
        F.when((F.col("transaction_type") == "purchase") & (F.col("channel").isin("Web", "POS")),
               F.concat(F.lit("4532xxxxxx"), F.lpad((F.rand() * 9999).cast("int"), 4, "0"))
        ).otherwise(F.lit(None)))

    # ip only present for online channels (Web/Mobile), never ATM/POS
    df = df.withColumn("ip_address",
        F.when(F.col("channel").isin("Web", "Mobile"),
               F.concat((F.rand() * 200 + 1).cast("int"), F.lit("."), (F.rand() * 200 + 1).cast("int"), F.lit(".1.10"))
        ).otherwise(F.lit(None)))

    df = df.withColumn("currency", F.lit("USD"))

    # lifecycle start state: everything lands PENDING or SETTLED same-day
    df = df.withColumn("status", F.when(F.rand() > 0.20, "SETTLED").otherwise("PENDING"))

    df = df \
        .withColumn("created_at", F.from_unixtime(
            F.unix_timestamp(F.lit(str(RUN_DATE)), "yyyy-MM-dd") + (F.rand() * 86400).cast("int")).cast("timestamp")) \
        .withColumn("updated_at", F.col("created_at")) \
        .drop("id")

    return df


def generate_transactions():
    df_accounts = spark.read.parquet(path("accounts"))
    df_merchants = spark.read.csv(path("merchants"), header=True)
    df_billers = spark.read.csv(path("billers"), header=True)

    if not IS_DAILY_INCREMENTAL:
        df_tx = _new_transactions(df_accounts, df_merchants, df_billers,
                                   NUM_INITIAL_TX_DAYS_BACKFILL, RUN_DATE_STR)
        df_tx.write.format("json").mode("overwrite").save(path("transactions"))
        return

    df_new = _new_transactions(df_accounts, df_merchants, df_billers, DAILY_TX_VOLUME, RUN_DATE_STR)

    # transition yesterday-and-older PENDING transactions to a final state
    df_pending_updates = None
    if landing_exists("transactions"):
        df_all = spark.read.json(path("transactions"))
        df_pending = df_all.filter(F.col("status") == "PENDING")
        if df_pending.take(1):
            df_pending_updates = df_pending \
                .withColumn("status", F.elt((F.rand() * 3 + 1).cast("int"),
                            F.lit("SETTLED"), F.lit("DECLINED"), F.lit("CANCELLED"))) \
                .withColumn("updated_at", F.current_timestamp())

    df_out = df_new if df_pending_updates is None else df_new.unionByName(df_pending_updates)
    df_out.write.format("json").mode("append").save(path("transactions"))


# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    generate_customers()
    generate_accounts()
    generate_merchants()
    generate_billers()
    generate_commission_rules()
    generate_transactions()
    print(f"Landing zone refreshed at {LANDING_PATH} for {RUN_DATE} "
          f"(incremental={IS_DAILY_INCREMENTAL})")